# Assignment 1: NumPy

Vectorised array operations and a timing comparison. AI-assisted draft: review the code and explanations before submission. Question 8 explicitly requests a Python loop, so that benchmark is the only exception to the general loop restriction.

In [1]:
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from time import perf_counter
np.random.seed(42)

## 1. Generate the array and compare storage

In [2]:
A = np.random.normal(50, 12, size=(1000, 5))
print("Shape:", A.shape, "dtype:", A.dtype, "bytes:", A.nbytes)
A = A.astype(np.float32)
print("After conversion:", A.dtype, "bytes:", A.nbytes)

Shape: (1000, 5) dtype: float64 bytes: 40000
After conversion: float32 bytes: 20000


## 2. Standardise columns

Keep A in its original units; use a separate standardised array.

In [3]:
standardised = (A - A.mean(axis=0, keepdims=True, dtype=np.float64)) / A.std(axis=0, keepdims=True, dtype=np.float64)
print("Means:", standardised.mean(axis=0).round(6))
print("Standard deviations:", standardised.std(axis=0).round(6))
assert np.allclose(standardised.mean(axis=0), 0, atol=2e-6)
assert np.allclose(standardised.std(axis=0), 1, atol=2e-6)

Means: [-0.  0.  0.  0.  0.]
Standard deviations: [1. 1. 1. 1. 1.]


## 3. Min-max scaling along either axis

In [4]:
def minmax(X, axis):
    low = np.min(X, axis=axis, keepdims=True)
    span = np.max(X, axis=axis, keepdims=True) - low
    return np.divide(X - low, span, out=np.zeros_like(X, dtype=float), where=span != 0)

column_scaled = minmax(A, axis=0)
row_scaled = minmax(A, axis=1)
print("Column minima:", column_scaled.min(axis=0))
print("Column maxima:", column_scaled.max(axis=0))
print("Row minima range:", row_scaled.min(axis=1).min(), row_scaled.min(axis=1).max())
print("Row maxima range:", row_scaled.max(axis=1).min(), row_scaled.max(axis=1).max())
assert np.allclose(column_scaled.min(axis=0), 0) and np.allclose(column_scaled.max(axis=0), 1)
assert np.allclose(row_scaled.min(axis=1), 0) and np.allclose(row_scaled.max(axis=1), 1)

Column minima: [0. 0. 0. 0. 0.]
Column maxima: [1. 1. 1. 1. 1.]
Row minima range: 0.0 0.0
Row maxima range: 1.0 1.0


## 4. Inject and fill missing values

Sampling without replacement makes exactly 5% of positions missing.

In [5]:
positions = np.random.choice(A.size, size=int(A.size * 0.05), replace=False)
A.flat[positions] = np.nan
print("Missing before:", np.isnan(A).sum())
missing = np.isnan(A)
column_means = np.nanmean(A, axis=0, keepdims=True)
A = np.where(missing, column_means, A)
print("Missing after:", np.isnan(A).sum())
assert np.isnan(A).sum() == 0

Missing before: 250
Missing after: 0


## 5. Replace outliers using column medians

In [6]:
outliers = np.abs(A - A.mean(axis=0, keepdims=True)) > 3 * A.std(axis=0, keepdims=True)
print("Replaced per column:", outliers.sum(axis=0))
A = np.where(outliers, np.median(A, axis=0, keepdims=True), A)

Replaced per column: [5 1 5 3 5]


## 6. Pairwise Euclidean distances

In [7]:
first = A[:200].astype(np.float64)
distances = np.sqrt(np.sum((first[:, None, :] - first[None, :, :]) ** 2, axis=2))
print("Shape:", distances.shape)
print("Zero diagonal:", np.allclose(np.diag(distances), 0))
print("Symmetric:", np.allclose(distances, distances.T))
assert np.allclose(np.diag(distances), 0)
assert np.allclose(distances, distances.T)

Shape: (200, 200)
Zero diagonal: True


Symmetric: True


## 7. Find five nearest neighbours

argpartition selects the five smallest distances without sorting all 200 entries in each row.

In [8]:
candidate_distances = distances.copy()
np.fill_diagonal(candidate_distances, np.inf)
nearest = np.argpartition(candidate_distances, kth=4, axis=1)[:, :5]
selected_distances = np.take_along_axis(candidate_distances, nearest, axis=1)
nearest = np.take_along_axis(nearest, np.argsort(selected_distances, axis=1), axis=1)
print("Shape:", nearest.shape, "dtype:", nearest.dtype)
print("First five rows:\n", nearest[:5])
assert nearest.shape == (200, 5)
assert not np.any(nearest == np.arange(200)[:, None])
assert np.all(np.max(np.take_along_axis(candidate_distances, nearest, axis=1), axis=1) <= np.partition(candidate_distances, 4, axis=1)[:, 4])

Shape: (200, 5) dtype: int64
First five rows:
 [[180 179 116 178 192]
 [114  17  31 112 197]
 [  3 158 132 145 102]
 [132   2 101   5 194]
 [160 158 143 102  43]]


## 8. Compare counting times

Both cells use %%time. perf_counter also records comparable operation timings so the speed-up can be calculated. The loop below is explicitly required by this question. A single timing varies with hardware and system load.

In [9]:
values = np.random.randint(0, 100, size=1_000_000)

In [10]:
%%time
start = perf_counter()
vector_counts = np.bincount(values, minlength=100)
vector_seconds = perf_counter() - start

CPU times: total: 15.6 ms
Wall time: 15.9 ms


In [11]:
%%time
start = perf_counter()
loop_counts = [0] * 100
for value in values:
    loop_counts[value] += 1
loop_seconds = perf_counter() - start

CPU times: total: 422 ms
Wall time: 442 ms


In [12]:
assert np.array_equal(vector_counts, loop_counts)
speedup = loop_seconds / vector_seconds
print(f"bincount: {vector_seconds:.6f} seconds")
print(f"Python loop: {loop_seconds:.6f} seconds")
print(f"Speed-up: {speedup:.2f}x")

bincount: 0.015284 seconds
Python loop: 0.427868 seconds
Speed-up: 27.99x


## 9. Sliding windows

In [13]:
series_values = np.random.normal(size=10_000)
windows = sliding_window_view(series_values, window_shape=10)
window_means = windows.mean(axis=1)
rolling_means = pd.Series(series_values).rolling(10).mean().iloc[9:].to_numpy()
print("Window shape:", windows.shape)
print("Matches pandas:", np.allclose(window_means, rolling_means))
assert windows.shape == (9991, 10)
assert np.allclose(window_means, rolling_means)

Window shape: (9991, 10)
Matches pandas: True


## 10. Reshape and average

The output has shape (4, 8): four groups with eight features each.
Each feature is the mean over the three rows belonging to its group.

In [14]:
original = np.arange(12 * 8).reshape(12, 8)
grouped = original.reshape(4, 3, 8)
group_means = grouped.mean(axis=1)
print("Result shape:", group_means.shape)
print(group_means)

Result shape: (4, 8)
[[ 8.  9. 10. 11. 12. 13. 14. 15.]
 [32. 33. 34. 35. 36. 37. 38. 39.]
 [56. 57. 58. 59. 60. 61. 62. 63.]
 [80. 81. 82. 83. 84. 85. 86. 87.]]


## 11. Save and reload the cleaned array

In [15]:
np.save("clean.npy", A)
loaded = np.load("clean.npy")
assert np.array_equal(loaded, A)
print("Reloaded array is identical:", np.array_equal(loaded, A))

Reloaded array is identical: True
